# 1 EDA
### 1. Instalando dependências

In [ ]:
#pip install pandas
#pip install numpy

### 1.1. Visão geral da tabela orders

In [2]:
import pandas as pd

ORDERS_CSV_DATA_PATH = '../data/1-lh_nautical_csv/orders.csv'

orders_df = pd.read_csv(ORDERS_CSV_DATA_PATH)

print(f"Quantidade total de linhas: {len(orders_df)}")
print(f"Quantidade total de colunas: {len(orders_df.columns)}")
print(f"Intervalo de datas analisado: {orders_df['created_at'].min()} a {orders_df['created_at'].max()}")

orders_df.head()

Quantidade total de linhas: 48998
Quantidade total de colunas: 13
Intervalo de datas analisado: 2020-01-01 01:19:28 a 2026-12-31 23:43:09


,id,order_number,channel,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,placed_at,created_at,updated_at
0,1,SO-000001,ecommerce,1136,NaN,1,paid,323.34,35.57,287.77,2022-09-06 05:37:37,2022-09-06 05:37:37,2022-09-06 05:37:37
1,2,SO-000002,ecommerce,618,9.0,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,2023-02-03 04:36:21,2023-02-03 04:36:21
2,3,SO-000003,pos,227,10.0,4,confirmed,17157.39,0.00,17157.39,2024-12-30 07:15:17,2024-12-30 07:15:17,2024-12-30 07:15:17
3,4,SO-000004,ecommerce,1123,NaN,3,paid,11095.62,665.74,10429.88,2024-03-12 20:39:36,2024-03-12 20:39:36,2024-03-12 20:39:36
4,5,SO-000005,ecommerce,426,9.0,2,confirmed,32842.03,0.00,32842.03,2020-03-27 13:52:12,2020-03-27 13:52:12,2020-03-27 13:52:12


### 1.2. Análise de valores numéricos

In [25]:
print(f"Valor máximo do campo 'total': {orders_df['total'].max()}")
print(f"Valor mínimo do campo 'total': {orders_df['total'].min()}")
print(f"Valor médio do campo 'total': {round(orders_df['total'].mean(), 2)}")

Valor máximo do campo 'total': 127262.02
Valor mínimo do campo 'total': 32.62
Valor médio do campo 'total': 28704.99


### 1.3. Interpretação da análise exploratória
#### 1.3.1. Identificação de possíveis outliers em na coluna `total`

A regra do intervalo interquartil (IQR) identifica valores estatisticamente atípicos dentro da própria distribuição da tabela `orders`. Essa regra sinaliza valores para investigação, mas não permite concluir, sozinha, que sejam erros.

`IQR = Q3 - Q1`, intervalo interquartil.

`limite_inferior = Q1 - 1,5 x IQR`, o que estiver abaixo é considerado um possível outlier.

`limite_superior = Q2 + 1,5 x IQR`, o que estiver acima é considerado um possível outlier.

In [20]:
q1 = orders_df['total'].quantile(0.25)
q3 = orders_df['total'].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outlier_condition = (orders_df['total'] < limite_inferior) | (orders_df['total'] > limite_superior)
outliers_total_df = orders_df.loc[outlier_condition].copy()

print(f'Q1: {q1:.2f}')
print(f'Q3: {q3:.2f}')
print(f'IQR: {iqr:.2f}')
print(f'Limite inferior: {limite_inferior:.2f}')
print(f'Limite superior: {limite_superior:.2f}')
print(f'Outliers em total: {len(outliers_total_df)} ({(len(outliers_total_df)/len(orders_df))*100:.2f}% das linhas)')

print("Os 5 menores outliers:")
print(outliers_total_df.nsmallest(5, 'total')[
    ['id', 'order_number', 'subtotal', 'discount_amount', 'total']
])

print("Os 5 maiores outliers:")
print(outliers_total_df.nlargest(5, 'total')[
    ['id', 'order_number', 'subtotal', 'discount_amount', 'total']
])

Q1: 13171.24
Q3: 40941.88
IQR: 27770.65
Limite inferior: -28484.74
Limite superior: 82597.85
Outliers em total: 452 (0.92% das linhas)
Os 5 menores outliers:
          id order_number  subtotal  discount_amount     total
2649    2707    SO-002707  82618.00              0.0  82618.00
20896  21352    SO-021352  82623.53              0.0  82623.53
4962    5068    SO-005068  82635.34              0.0  82635.34
31779  32448    SO-032448  82650.93              0.0  82650.93
7131    7283    SO-007283  82699.84              0.0  82699.84
Os 5 maiores outliers:
          id order_number   subtotal  discount_amount      total
24584  25100    SO-025100  127262.02             0.00  127262.02
24299  24810    SO-024810  125273.92             0.00  125273.92
22858  23351    SO-023351  118037.91             0.00  118037.91
30628  31266    SO-031266  118676.15          4747.05  113929.10
34339  35048    SO-035048  113622.80             0.00  113622.80


Como o valor do limite inferior é negativo e não há valores negativos para a coluna `total`, todos os possíveis outliers encontrados estão presentes acima do limite superior.

#### 1.3.2. Qualidade e consistência dos dados

In [32]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48998 entries, 0 to 48997
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               48998 non-null  int64  
 1   order_number     48998 non-null  str    
 2   channel          48998 non-null  str    
 3   customer_id      48998 non-null  int64  
 4   salesperson_id   24867 non-null  float64
 5   location_id      48998 non-null  int64  
 6   status           48998 non-null  str    
 7   subtotal         48998 non-null  float64
 8   discount_amount  48998 non-null  float64
 9   total            48998 non-null  float64
 10  placed_at        48998 non-null  str    
 11  created_at       48998 non-null  str    
 12  updated_at       48998 non-null  str    
dtypes: float64(4), int64(3), str(6)
memory usage: 4.9 MB


É possível observar que as datas estão no formato de str e há valores nulos para a coluna salesperson_id.

In [41]:
resumo_nulos = pd.DataFrame({
    'quantidade_nulos': orders_df.isna().sum(),
    'percentual_nulos': orders_df.isna().mean().mul(100).round(2)
})

print('Colunas com valores nulos:')
display(resumo_nulos[resumo_nulos['quantidade_nulos'] > 0])

print(f'IDs duplicados: {orders_df["id"].duplicated().sum()}')
print(f'Números de pedido duplicados: {orders_df["order_number"].duplicated().sum()}')
print(f'Linhas inteiramente duplicadas: {orders_df.duplicated().sum()}')

Colunas com valores nulos:


,quantidade_nulos,percentual_nulos
salesperson_id,24131,49.25


IDs duplicados: 0
Números de pedido duplicados: 0
Linhas inteiramente duplicadas: 0


In [ ]:
print('Nulos em salesperson_id por canal:')
display(pd.crosstab(
    orders_df['channel'],
    orders_df['salesperson_id'].isna(),
    rownames=['channel'],
    colnames=['salesperson_id nulo']
))

Nulos em salesperson_id por canal:


salesperson_id nulo,False,True
channel,,
ecommerce,10211,24131
pos,14656,0


Conforme evidênciado acima, os nulos presentes na coluna salesperson_id aparentemente fazem parte da regra de negócio onde o canal de vendas presencial precisa de um vendedor e o canal de vendas ecommerce pode ou não haver um vendedor.

In [46]:
diferenca_total = (
    orders_df['subtotal'] - orders_df['discount_amount'] - orders_df['total']
).abs()

colunas_monetarias = ['subtotal', 'discount_amount', 'total']

print(f'Totais incompatíveis com subtotal - desconto: {(diferenca_total > 0.01).sum()}') # 0,01 por conta do float
print(f"Total maior que subtotal: {(orders_df['total'] > orders_df['subtotal']).sum()}")
print(f'Descontos maiores que o subtotal: {(orders_df["discount_amount"] > orders_df["subtotal"]).sum()}')
print(f'Valores monetários negativos: {(orders_df[colunas_monetarias] < 0).sum().sum()}')
print(f'Canais encontrados: {sorted(orders_df["channel"].unique())}')
print(f'Status encontrados: {sorted(orders_df["status"].unique())}')

Totais incompatíveis com subtotal - desconto: 0
Total maior que subtotal: 0
Descontos maiores que o subtotal: 0
Valores monetários negativos: 0
Canais encontrados: ['ecommerce', 'pos']
Status encontrados: ['cancelled', 'confirmed', 'draft', 'paid']


In [49]:
colunas_data = ['placed_at', 'created_at', 'updated_at']
datas_convertidas = orders_df[colunas_data].apply(pd.to_datetime, errors='coerce')

print('Datas inválidas por coluna:')
display(datas_convertidas.isna().sum().to_frame('quantidade'))
print(f'Atualizações anteriores à criação: {(datas_convertidas["updated_at"] < datas_convertidas["created_at"]).sum()}')

Datas inválidas por coluna:


,quantidade
placed_at,0
created_at,0
updated_at,0


Atualizações anteriores à criação: 0


## Parte 3 - Interpretação

Com base apenas na tabela `orders`, conclui-se que ela apresenta boa consistência estrutural e monetária, mas requer algumas validações antes de determinadas análises. Pela regra do IQR, foram identificados **452 possíveis outliers na coluna `total` reprensentando 0,92% dos pedidos**, acima de 82.597,85 (limite superior), com máximo de 127.262,02. Dentro da própria tabela, esses registros respeitam a relação `total = subtotal - discount_amount` e não apresentam valores negativos. Portanto, são valores estatisticamente atípicos, mas não é possível classificá-los como erros usando somente `orders`. Eles não devem ser removidos automaticamente e devem ser considerados em métricas sensíveis a extremos, como a média. Além disso, as colunas `placed_at`, `created_at` e `updated_at` foram inicialmente importadas como texto (`str`) e devem ser convertidas para o tipo `datetime` antes de análises temporais.

Quanto à qualidade, não há nulos nas variáveis monetárias ou identificadoras, nem IDs, números de pedido ou linhas duplicadas. O único campo com ausências é `salesperson_id`: **24.131 registros (49,25%)**, todos do canal ecommerce. Esse padrão sugere que o preenchimento pode ser opcional nesse canal, mas a regra de negócio deve ser confirmada antes de análises por vendedor. Também não foram encontrados valores monetários negativos, descontos superiores ao subtotal, erros na fórmula do total ou datas inválidas.

Portanto, a tabela está suficientemente consistente para análises exploratórias gerais, mas não deve ser usada de forma indiscriminada sem tratamento prévio. Recomenda-se validar a regra de preenchimento de `salesperson_id`.